# Online User Requirement Model Selection

**Purpose:** Accept a user's natural-language requirement and recommend the Top-3 PaperModel / OPFModel instances from the `OPF_Knowledge_Graph2.0.owl` knowledge graph.

**Pipeline:**

1. Read all model instances and run the LLM prefilter in groups of 12.
2. Merge all candidates with a prefilter score of at least 65.
3. Give the LLM every candidate's complete data and object properties for close reading, independent scoring, and ranking.
4. Verify that close reading is complete for every candidate, then select the final Top-3.

This workflow no longer performs a separate consistency check or truncates the candidate set before close reading.

## 1. Environment and Path Setup

Import the project paths and packaged function library. This section only prepares the environment; it does not call the LLM or write OWL.


In [ ]:
# Project package bootstrap.
# Reusable functions are imported from src/opfmodelselection instead of being defined in notebook cells.

import sys
from pathlib import Path

_CWD = Path.cwd().resolve()
for _candidate in [_CWD, _CWD.parent]:
    if (_candidate / "src" / "opfmodelselection").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise RuntimeError("Cannot find src/opfmodelselection from current working directory.")

from src.opfmodelselection.config import *
print("项目配置已加载 / Project config loaded:", PROJECT_ROOT)


## 1.1 Shared LLM/API Configuration

Load the shared LLM/API configuration. The model name, temperature, and OpenAI/RWTH route are maintained centrally in `src/opfmodelselection/config.py`.


In [ ]:
# Shared LLM/API configuration is imported from src.opfmodelselection.config.
print_llm_config_status()


## 2. Load the OPF Schema and Knowledge Graph XML

Read the following files:

- `OPF_Ontology2.0.owl`: classes, properties, and the taxonomy schema.
- `OPF_Knowledge_Graph2.0.owl`: ingested PaperModel / OPFModel instances.

Online recommendation reads only these two files; it does not load the complete OEO/CIM closure or write OWL.


In [ ]:
# Lightweight ontology file check.
check_project_paths(include_oeo_model=True)
OPF_XML_TEXT, OPF_XML_ROOT, OPF_OWL_IMPORTS = read_opf_xml()
print_opf_xml_status()
print("已跳过完整 Owlready2 加载 / Skipped full Owlready2 load for this online recommendation notebook.")


## 9. Direct Online Recommendation

The recommendation workflow directly compares the user's original requirement with the complete information of ingested PaperModel instances, without taxonomy whitelist mapping.

Key intermediate results:

- `online_prefilter_result`: grouped prefilter results and evidence packages.
- `online_prefilter_candidates`: all merged candidates entering close reading, with complete PaperModel information.
- `online_close_reading_result`: close-reading scores, analyses, and rankings for all candidates.
- `online_final_top3`: the final Top-3 selected from the complete close-reading results.

### 9.1 Function Imports

Import the recommendation functions from `online_recommendation.py`. The notebook orchestrates the workflow without defining recommendation logic.


In [ ]:
# Online user-requirement recommendation functions.
from src.opfmodelselection.online_recommendation import *
from src.opfmodelselection.user_requirements import USER_REQUIREMENTS_BY_ID

print("在线推荐函数已加载 / Direct online recommendation functions loaded.")
print("按顺序运行 9.2 -> 9.6，或直接调用 run_direct_online_recommendation_pipeline(motivation_text)。")

### 9.2 Set the User Requirement

Select a `requirement_id` from the canonical table of 54 requirements, or provide `custom_requirement_text`. The resulting `motivation_text` is the single user-requirement text used by both grouped prefiltering and complete close reading.

In [ ]:
# 9.2 Select one canonical requirement (R01-R19 or M01-M35), or provide custom text.

selected_requirement_id = "R03"
custom_requirement_text = ""
motivation_text = (
    custom_requirement_text.strip()
    or USER_REQUIREMENTS_BY_ID[selected_requirement_id]["requirement_text"]
)

print("用户原始需求 / User original requirement:")
print(motivation_text.strip())


### 9.3 LLM Grouped Prefilter

Run the LLM prefilter: read all PaperModel instances, evaluate them in groups of 12, and retain candidates with `prefilter_score >= 65`.

Outputs include candidate models, prefilter scores, reasons for matching, risks or gaps, and supporting evidence.


In [ ]:
# 9.3 Run grouped LLM prefilter and print merged candidates with evidence packages.

if "motivation_text" not in globals() or not str(motivation_text).strip():
    raise RuntimeError("请先运行 9.2 设置 motivation_text / Run 9.2 first.")

online_prefilter_result = run_llm_grouped_prefilter(
    user_requirement_text=motivation_text,
    group_size=ONLINE_PREFILTER_GROUP_SIZE,
)

print("\n合并后的候选模型证据包 / Merged candidate evidence packages")
print("=" * 100)
for index, candidate in enumerate(online_prefilter_result.get("selected_candidates", []), start=1):
    print("\n" + "-" * 100)
    print(f"{index}. {candidate['instance_name']}")
    print("初筛分 / prefilter_score:", candidate.get("prefilter_score"))
    print("为什么匹配 / why_matched:", candidate.get("why_matched"))
    print("主要风险或缺口 / main_risk_or_gap:", candidate.get("main_risk_or_gap"))
    print("支持证据 / supporting_evidence:")
    for evidence in candidate.get("supporting_evidence", []):
        print("  -", evidence)

print("\n结构化初筛结果 / Structured prefilter result:")
print(json.dumps(online_prefilter_result, indent=2, ensure_ascii=False))


### 9.4 Merge All Prefilter Candidates

Read the merged grouped-prefilter results and enrich every candidate scoring at least 65 with its complete data and object properties from the knowledge graph. This step neither calls the LLM nor selects a Top-N subset.

In [ ]:
# 9.4 Enrich every merged prefilter candidate with its complete PaperModel record.
if "online_prefilter_result" not in globals():
    raise RuntimeError("请先运行 9.3 生成 online_prefilter_result / Run 9.3 first.")

online_prefilter_candidates = build_close_reading_candidates_from_prefilter(
    online_prefilter_result
)

print("进入精读的候选数量 / Candidate count entering close reading:", len(online_prefilter_candidates))
for index, candidate in enumerate(online_prefilter_candidates, start=1):
    evidence = candidate["evidence_package"]
    print(
        f"{index}. {candidate['kg_info']['instance_name']} "
        f"| prefilter_score={evidence.get('prefilter_score')}"
    )

### 9.5 LLM Close Reading of Every Candidate

Give the LLM the complete PaperModel information for every candidate from Section 9.4 so that it independently produces a `final_score`, recommendation reason, matched needs, strengths, limitations, and a concise model explanation.

The program verifies that complete close-reading content is returned for every input candidate. Missing results trigger automatic retries; if the results remain incomplete, Top-3 generation stops rather than mixing candidates without close-reading results into the final output.

In [ ]:
# 9.5 Close-read, score, and order every prefilter candidate.
if "online_prefilter_candidates" not in globals():
    raise RuntimeError("请先运行 9.4 生成 online_prefilter_candidates / Run 9.4 first.")
if not online_prefilter_candidates:
    raise RuntimeError("没有通过初筛的候选模型 / No candidate passed prefilter.")

online_raw_close_reading = call_llm_close_reading(
    motivation_text,
    online_prefilter_candidates,
)
online_close_reading_result = validate_close_reading_output(
    online_raw_close_reading,
    online_prefilter_candidates,
)

print("完整精读结果 / Complete close-reading results")
for row in online_close_reading_result["ranked_candidates"]:
    print(
        f"{row['rank']}. {row['instance_name']} "
        f"| final_score={row['final_score']}"
    )
    print("   推荐理由 / Recommendation reason:", row["recommendation_reason"])
    print("   匹配点 / Matched needs:", row["matched_needs"])
    print("   优点 / Strengths:", row["strengths"])
    print("   不足 / Limitations:", row["limitations"])
    print("   模型简述 / Model explanation:", row["simple_model_explanation"])

### 9.6 Final Top-3 Recommendation

Select the Top-3 from the complete close-reading results ordered by `final_score`. This step neither calls the LLM nor writes OWL.

In [ ]:
# 9.6 Select and print the final Top-3 from complete close-reading results.
if "online_close_reading_result" not in globals():
    raise RuntimeError("请先运行 9.5 生成 online_close_reading_result / Run 9.5 first.")

online_final_top3 = online_close_reading_result["ranked_candidates"][:ONLINE_FINAL_TOP_N]
expected_count = min(ONLINE_FINAL_TOP_N, len(online_prefilter_candidates))
if len(online_final_top3) != expected_count:
    raise RuntimeError(
        f"Top-3 数量校验失败：期望 {expected_count}，实际 {len(online_final_top3)}。"
    )

print("最终推荐 Top-3 / Final recommended Top-3")
for row in online_final_top3:
    print(f"Top {row['rank']}: {row['instance_name']} | final_score={row['final_score']}")
    print("   推荐理由 / Recommendation reason:", row["recommendation_reason"])
    print("   匹配点 / Matched needs:", row["matched_needs"])
    print("   优点 / Strengths:", row["strengths"])
    print("   不足 / Limitations:", row["limitations"])
    print("   模型简述 / Model explanation:", row["simple_model_explanation"])

## 10. Validation Moved

Validation of the current recommendation mechanism and semantic validation of paper ingestion are located in `src/Validation2.0.ipynb`.
